# Mains-dose check — the supply voltage sets the per-device draw

Assertion guard for the dose-by-country arithmetic quoted in the IEEE paper
(Section V-B, Section VIII, and the EMC-filter figure captions in both tiers)
and in the EEA 2026 FSR paper (sections 4.3 and 5.4). One worldwide fleet,
capacitance chosen once; the mains it lands on sets the standing draw as
Q = omega * C * V^2. Every figure quoted in prose is asserted here; if a
quoted value changes, change it here first and re-run (tex header rule).

Run:  python src/mains_dose_check.py   (prints the table, exits nonzero on
any mismatch between the computed physics and the values quoted in prose).

In [1]:
import math


def var_per_uf(volts: float, hertz: float) -> float:
    """Standing draw of one microfarad across the given mains, in VAr."""
    return 2 * math.pi * hertz * 1e-6 * volts**2


# The three mains the prose quotes. Japan runs both 50 Hz (east) and 60 Hz
# (west) at 100 V, hence a range rather than a point.
NZ_EU = var_per_uf(230, 50)      # New Zealand / Europe / most of the world
NA = var_per_uf(120, 60)         # North America
JP_50 = var_per_uf(100, 50)      # Japan, eastern grid
JP_60 = var_per_uf(100, 60)      # Japan, western grid

print(f"VAr per microfarad:  230 V 50 Hz {NZ_EU:.2f} | 120 V 60 Hz {NA:.2f} "
      f"| 100 V {JP_50:.2f}-{JP_60:.2f}")
print(f"Ratios to 230 V:     North America x{NZ_EU/NA:.2f} | "
      f"Japan x{NZ_EU/JP_60:.2f}-{NZ_EU/JP_50:.2f}")

# Prose: "16.6 VAr per microfarad" at 230 V, 50 Hz (V-B and App B, both tiers).
assert round(NZ_EU, 1) == 16.6, NZ_EU
# Prose: "5.4 VAr" on North America's 120 V, 60 Hz mains.
assert round(NA, 1) == 5.4, NA
# Prose: "3.1--3.8 VAr" at Japan's 100 V (50 and 60 Hz grids).
assert round(JP_50, 1) == 3.1 and round(JP_60, 1) == 3.8, (JP_50, JP_60)
# Prose: "roughly three times" North America's per-device draw (3.06).
assert round(NZ_EU / NA, 2) == 3.06, NZ_EU / NA
# Prose: "four to five times" Japan's (4.4--5.3).
assert 4.35 <= NZ_EU / JP_60 <= 4.45 and 5.25 <= NZ_EU / JP_50 <= 5.35, \
    (NZ_EU / JP_60, NZ_EU / JP_50)
# Prose ordering (V-B close and VIII): omega*V^2 orders Japan < NA < 230-V world.
assert JP_50 < JP_60 < NA < NZ_EU

VAr per microfarad:  230 V 50 Hz 16.62 | 120 V 60 Hz 5.43 | 100 V 3.14-3.77
Ratios to 230 V:     North America x3.06 | Japan x4.41-5.29


## The X-capacitor range on each mains
The figure captions quote the 0.1--2.2 uF X-capacitance range as 1.7--37 VAr
at 230 V and 0.5--12 VAr on North America's mains.

In [2]:
for lo_c, hi_c in [(0.1, 2.2)]:
    print(f"{lo_c}-{hi_c} uF at 230 V 50 Hz: {lo_c*NZ_EU:.1f}-{hi_c*NZ_EU:.1f} VAr"
          f" | at 120 V 60 Hz: {lo_c*NA:.2f}-{hi_c*NA:.1f} VAr")
    # Prose/captions: "1.7--37 VAr at 230 V" (both tiers, V-B text and figure).
    assert round(lo_c * NZ_EU, 1) == 1.7 and round(hi_c * NZ_EU) == 37
    # Captions: "0.5--12 VAr" on 120 V, 60 Hz mains.
    assert round(lo_c * NA, 1) == 0.5 and round(hi_c * NA) == 12

# The V-B spot values at 230 V, 50 Hz: 0.1, 0.47, 1, 2.2 uF -> 1.7, 7.8,
# 16.6, 36.6 VAr (pre-existing prose, asserted for completeness).
for c, expected in [(0.1, 1.7), (0.47, 7.8), (1.0, 16.6), (2.2, 36.6)]:
    assert round(c * NZ_EU, 1) == expected, (c, c * NZ_EU)

0.1-2.2 uF at 230 V 50 Hz: 1.7-36.6 VAr | at 120 V 60 Hz: 0.54-11.9 VAr


## Per-device-class dose table (extended tier, Appendix B)
Census of typical total line-to-neutral (class X) capacitance by device
class, from the device-fleet research record (reference designs with
published BOMs, filter datasheets, teardowns —
research/device-fleet/DEVICE_FLEET_STANDING_REACTIVE.md section 2.3), and
the standing draw each range takes on the two mains, exactly as printed in
the appendix table. The capacitance ranges are the data; both VAr columns
are derived here and must not be edited in the tex without re-running.
(One cell corrects the research record's own table: 0.37 uF at 230 V is
6.1 VAr at 1 dp, not the 6.2 carried there.) Classes with no X-capacitor
(chargers <= 20 W, ~5 W LED retrofit lamps) appear in the table as
zero rows with no arithmetic to guard.

In [3]:
# (device class, C_lo uF, C_hi uF, printed VAr at 230 V/50 Hz, at 120 V/60 Hz)
DEVICE_CLASSES = [
    ("USB-PD and laptop supplies 27-65 W",  0.10, 0.22, (1.7, 3.7), (0.5, 1.2)),
    ("Laptop and retail supplies 65-140 W", 0.22, 0.47, (3.7, 7.8), (1.2, 2.6)),
    ("LED driver 12-100 W",                 0.066, 0.45, (1.1, 7.5), (0.4, 2.4)),
    ("TV / monitor supply 100-220 W",       0.44, 0.66, (7.3, 11.0), (2.4, 3.6)),
    ("Desktop computer supply",             0.37, 1.80, (6.1, 29.9), (2.0, 9.8)),
    ("Whiteware (laundry, inverter refrigeration)",
                                            0.47, 1.41, (7.8, 23.4), (2.6, 7.7)),
    ("Heat-pump / AC outdoor unit",         1.41, 2.00, (23.4, 33.2), (7.7, 10.9)),
    ("Single-phase VSD input filter",       0.94, 4.2, (15.6, 69.8), (5.1, 22.8)),
    ("Generic single-phase appliance filter",
                                            0.10, 1.0, (1.7, 16.6), (0.5, 5.4)),
]
for name, lo, hi, (a230, b230), (a120, b120) in DEVICE_CLASSES:
    assert round(lo * NZ_EU, 1) == a230 and round(hi * NZ_EU, 1) == b230, \
        (name, lo * NZ_EU, hi * NZ_EU)
    assert round(lo * NA, 1) == a120 and round(hi * NA, 1) == b120, \
        (name, lo * NA, hi * NA)
    print(f"{name:46s} {lo}-{hi} uF | {a230}-{b230} VAr at 230/50 "
          f"| {a120}-{b120} VAr at 120/60")

# Section V-B's coarser prose rounding of the heat-pump row (both tiers):
# "1.4--2.0 uF each (23--33 VAr)" — same physics, integer print.
assert round(1.41 * NZ_EU) == 23 and round(2.00 * NZ_EU) == 33

print("All quoted mains-dose figures verified against the physics.")

USB-PD and laptop supplies 27-65 W             0.1-0.22 uF | 1.7-3.7 VAr at 230/50 | 0.5-1.2 VAr at 120/60
Laptop and retail supplies 65-140 W            0.22-0.47 uF | 3.7-7.8 VAr at 230/50 | 1.2-2.6 VAr at 120/60
LED driver 12-100 W                            0.066-0.45 uF | 1.1-7.5 VAr at 230/50 | 0.4-2.4 VAr at 120/60
TV / monitor supply 100-220 W                  0.44-0.66 uF | 7.3-11.0 VAr at 230/50 | 2.4-3.6 VAr at 120/60
Desktop computer supply                        0.37-1.8 uF | 6.1-29.9 VAr at 230/50 | 2.0-9.8 VAr at 120/60
Whiteware (laundry, inverter refrigeration)    0.47-1.41 uF | 7.8-23.4 VAr at 230/50 | 2.6-7.7 VAr at 120/60
Heat-pump / AC outdoor unit                    1.41-2.0 uF | 23.4-33.2 VAr at 230/50 | 7.7-10.9 VAr at 120/60
Single-phase VSD input filter                  0.94-4.2 uF | 15.6-69.8 VAr at 230/50 | 5.1-22.8 VAr at 120/60
Generic single-phase appliance filter          0.1-1.0 uF | 1.7-16.6 VAr at 230/50 | 0.5-5.4 VAr at 120/60
All quoted mains-dose f